In [24]:
"""
=============================================================
=== MULTITHREADING IN PYTHON - A COMPLETE REVISION GUIDE ===
=============================================================

This file contains a summary of key multithreading concepts,
from basic execution to advanced thread-safe practices.

-------------------------------------------------------------
### CONCEPT 1: The Baseline - Sequential Execution & Timing
-------------------------------------------------------------
Before optimizing with threads, we need a baseline.

- `time.perf_counter()`: A high-precision timer. We call it before and after
  our code to measure how long it took.
  
- `time.sleep(seconds)`: Pauses the entire program's execution. We use this
  to simulate a real task that takes time to complete (like a network
  download). This is known as an "I/O-bound" operation because the program
  is waiting for Input/Output.

# Code Example:
import time
start_time = time.perf_counter()
time.sleep(2) # Simulate a 2-second task
end_time = time.perf_counter()
# print(f"Task took {end_time - start_time:.2f} seconds")


--------------------------------------------------------------------
### CONCEPT 2: Main Thread vs. Worker Threads (A KEY CONCEPT)
--------------------------------------------------------------------
Every Python script runs in a "Main Thread". We can create new "Worker
Threads" to perform tasks concurrently.

- `threading.Thread(target=my_function)`: Creates a new thread object.
  - `target`: The function the worker thread will execute.

- `thread.start()`: This is a NON-BLOCKING call. It's like a manager
  handing a task to a worker and immediately walking away without waiting
  for the worker to finish.

*** STICKY POINT REVISITED ***
The "Early Finish" Problem: Because `.start()` is non-blocking, the Main
Thread will kick off the worker threads and then immediately continue with its
own work. If the Main Thread has no more work to do, it will finish and print
its final messages *before* the long-running worker threads have completed
their tasks.

# Code Example (The Problem):
import threading

def long_task():
    time.sleep(2)
    print("Worker thread finished.")

t1 = threading.Thread(target=long_task)
t1.start() # Main thread tells t1 to start, and immediately moves on
print("Main thread finished.")

# OUTPUT ORDER:
# Main thread finished.
# (2 seconds later)
# Worker thread finished.


--------------------------------------------------------------------
### CONCEPT 3: The Solution - `thread.join()`
--------------------------------------------------------------------
To solve the "early finish" problem, we must tell the Main Thread to wait.

- `thread.join()`: This is a BLOCKING call. It tells the thread that called
  it (usually the Main Thread) to "PAUSE and wait right here until this
  worker thread is completely finished."

*** STICKY POINT REVISITED ***
Using our analogy, `.join()` is like the manager standing at the office door,
refusing to go home until every worker has come back and reported that their
task is complete.

# Code Example (The Solution):
t1 = threading.Thread(target=long_task)
t1.start()
t1.join() # Main thread PAUSES HERE until t1 is done.
print("Main thread finished AFTER the worker.")

# OUTPUT ORDER:
# (2 seconds later)
# Worker thread finished.
# Main thread finished AFTER the worker.


--------------------------------------------------------------------
### CONCEPT 4: Managing Many Threads with Loops
--------------------------------------------------------------------
To run many threads, we use a standard pattern: two separate loops.

1. A loop to START all the threads.
2. A loop to JOIN all the threads.

It is CRITICAL to use two separate loops. If you start and join in the
same loop, the program becomes sequential again, and you lose all the
benefits of threading.

# Code Example (The Correct Pattern):
threads = []
for _ in range(5):
    t = threading.Thread(target=long_task)
    t.start()       # Start the thread
    threads.append(t) # And add it to our list to track it

# This first loop finishes very quickly, starting all 5 threads.

for thread in threads:
    thread.join() # Now, this second loop waits for each one to finish.

# Total time is ~2 seconds, not 10.


-----------------------------------------------------------------------------
### CONCEPT 5: The Modern Way - `concurrent.futures.ThreadPoolExecutor`
-----------------------------------------------------------------------------
This is a high-level, modern way to manage threads that is much cleaner
and safer. It's like hiring an expert manager to handle the details.

*** STICKY POINT REVISITED: The `with` statement ***
The `with` statement creates a "context manager". It handles setup and,
more importantly, AUTOMATIC cleanup. When the `with` block is exited,
it automatically and safely shuts down the thread pool, which includes
waiting for all submitted tasks to complete (an implicit `.join()`).

*** STICKY POINT REVISITED: `executor.map()` ***
The `.map()` method is a powerful command you give to the executor. It
replaces the manual loops for creating and starting threads.

`executor.map(function, data_list)` tells the manager:
"Take this `function` and apply it to every single item in the `data_list`,
running each one in a separate thread from your pool."

# Code Example (The Modern Pattern):
import concurrent.futures

urls = ["url1", "url2", "url3"] # Example data

def download_file(url):
    # ... download logic ...
    print(f"Finished downloading {url}")

# This block replaces BOTH the start loop and the join loop.
with concurrent.futures.ThreadPoolExecutor() as executor:
    executor.map(download_file, urls)

print("All downloads are complete.")


-----------------------------------------------------------------------------
### CONCEPT 6: Shared State, Race Conditions, and Locks
-----------------------------------------------------------------------------
This is the most complex topic. A "race condition" is a bug that occurs
when multiple threads try to modify a shared variable at the same time,
leading to incorrect results.

The solution is a `threading.Lock`.

- `threading.Lock()`: Creates a lock object. A lock can be either
  "locked" or "unlocked".

*** STICKY POINT REVISITED: The `with lock:` block ***
This creates a "critical section". Only one thread can "acquire" the lock
and enter this block of code at a time. All other threads that arrive must
wait until the first one is finished and releases the lock.

Analogy: The lock is a key to a single-person bathroom. Only one thread can
have the key and be inside at a time. Others must wait outside.

# Code Example (Thread-Safe Counter):
shared_counter = 0
counter_lock = threading.Lock() # The bathroom key

def safe_increment():
    global shared_counter
    with counter_lock: # Only one thread can be in here at a time
        # This is the "critical section" or the "bathroom"
        current_value = shared_counter
        time.sleep(0.1) # Simulate some work
        shared_counter = current_value + 1

# If you run safe_increment with multiple threads, the final value of
# shared_counter will always be correct because the lock prevents the
# race condition.

"""

'\n=============================================================\n=== MULTITHREADING IN PYTHON - A COMPLETE REVISION GUIDE ===\n=============================================================\n\nThis file contains a summary of key multithreading concepts,\nfrom basic execution to advanced thread-safe practices.\n\n-------------------------------------------------------------\n### CONCEPT 1: The Baseline - Sequential Execution & Timing\n-------------------------------------------------------------\nBefore optimizing with threads, we need a baseline.\n\n- `time.perf_counter()`: A high-precision timer. We call it before and after\n  our code to measure how long it took.\n  \n- `time.sleep(seconds)`: Pauses the entire program\'s execution. We use this\n  to simulate a real task that takes time to complete (like a network\n  download). This is known as an "I/O-bound" operation because the program\n  is waiting for Input/Output.\n\n# Code Example:\nimport time\nstart_time = time.perf_counter()

In [1]:
import time
start = time.perf_counter()

def test_fun():
    print("do something")
    print("sleep for 2 sec")
    time.sleep(2)
    print("done with sleeping")

test_fun()
end = time.perf_counter()
print(f"the program finished in {round(end-start, 2)} second")

do something
sleep for 2 sec
done with sleeping
the program finished in 2.0 second


In [2]:
import time
start = time.perf_counter()

def test_fun():
    print("do something")
    print("sleep for 2 sec")
    time.sleep(2)
    print("done with sleeping")

test_fun()
test_fun()
test_fun()
test_fun()
test_fun()
end = time.perf_counter()
print(f"the program finished in {round(end-start, 2)} second")

do something
sleep for 2 sec
done with sleeping
do something
sleep for 2 sec
done with sleeping
do something
sleep for 2 sec
done with sleeping
do something
sleep for 2 sec
done with sleeping
do something
sleep for 2 sec
done with sleeping
the program finished in 10.01 second


In [3]:
# in above : since the program ran sequentially(single thread a single core) so it took 10 seconds 

In [4]:
import time
import threading
start = time.perf_counter()

def test_fun():
    print("do something")
    print("sleep for 2 sec")
    time.sleep(2)
    print("done with sleeping")

# run the program on threads 
t1 = threading.Thread(target=test_fun)
t2 = threading.Thread(target=test_fun)

t1.start() # to start the thread 
t2.start()

end = time.perf_counter()
print(f"the program finished in {round(end-start, 2)} second")

do something
sleep for 2 sec
do something
sleep for 2 sec
the program finished in 0.01 second


done with sleeping
done with sleeping


In [ ]:
"""
to start these two tread t1 and t2 there is a main thread. 
So main thread starts t1 and t2 , and all other process except t1 and t2 will be executed by main thread 
-> before this t1 and t2 could be completed, the main thread got executed and we got : 
        print(f"the program finished in {round(end-start, 2)} second")   earlier 



"""

In [ ]:
# to avoid this : the print statement executed in last , we will use join()

import time
import threading
start = time.perf_counter()

def test_fun():
    print("do something")
    print("sleep for 2 sec")
    time.sleep(2)
    print("done with sleeping")

# run the program on threads 
t1 = threading.Thread(target=test_fun)
t2 = threading.Thread(target=test_fun)

t1.start() # to start the thread 
t2.start()

t1.join() #  join first executed these t1 , t2 , and then the main thread will be executed 
t2.join()

end = time.perf_counter()
print(f"the program finished in {round(end-start, 2)} second")

do something
sleep for 2 sec
do something
sleep for 2 sec
done with sleeping
done with sleeping
the program finished in 2.01 second


In [19]:
import time
import threading
start = time.perf_counter()

def test_fun():
    print("do something")
    print("sleep for 2 sec")
    time.sleep(2)
    print("done with sleeping")

threads = []
for i in range(5):
    print(i)
    t = threading.Thread(target= test_fun)
    t.start()
    threads.append(t)

for thread in threads:
    thread.join()

end = time.perf_counter()
print(f"the program finished in {round(end-start, 2)} second")

0
do something
sleep for 2 sec
1
do something
sleep for 2 sec
2
do something
sleep for 2 sec
3
do something
sleep for 2 sec
4
do something
sleep for 2 sec
done with sleeping
done with sleeping
done with sleeping
done with sleeping
done with sleeping
the program finished in 2.0 second


In [7]:
# in above example : it get completed in 2 seconds instead of 20 sencond 

In [21]:
#Using multithreading with function that takes an argument 
import time
import threading
start = time.perf_counter()

def test_fun(args):
    print("do something")
    print(f"sleep for {args} sec")
    time.sleep(args)
    print("done with sleeping")

threads = []
for i in range(10):
    t = threading.Thread(target= test_fun, args= [1]) # passing argument
    t.start()
    threads.append(t)

for thread in threads:
    thread.join()

end = time.perf_counter()
print(f"the program finished in {round(end-start, 2)} second")

do something
sleep for 1 sec
do something
sleep for 1 sec
do something
sleep for 1 sec
do something
sleep for 1 sec
do something
sleep for 1 sec
do something
sleep for 1 sec
do something
sleep for 1 sec
do something
sleep for 1 sec
do something
sleep for 1 sec
do something
sleep for 1 sec
done with sleeping
done with sleeping
done with sleeping
done with sleeping
done with sleeping
done with sleeping
done with sleeping
done with sleeping
done with sleeping
done with sleeping
the program finished in 1.01 second


In [9]:
# use case : 
# Multithreadin works well with I/O bound task means where same output has to wait for input 
# eg: reading writing files , database querry 
   

In [ ]:
# how we can use multithreading for downloads : 
import time 
import threading

start = time.perf_counter()

url_list = [
    "https://www.w3.org/WAI/ER/tests/xhtml/testfiles/resources/pdf/dummy.pdf",     # Small PDF file
    "https://file-examples.com/storage/fe5c9c62f/file-example_TXT_10kB.txt",       # 10KB TXT file
    "https://file-examples.com/storage/fe6d74a6/file-example_PNG_500kB.png"        # Small PNG image
]

data_list = ["dummy.pdf", "sample.txt", "image.png"]


import urllib.request
def file_download(url, filename):
    try:
        urllib.request.urlretrieve(url, filename)
        print(f"Downloaded: {filename}")
    except Exception as e:
        print(f"Error downloading {url}: {e}")

threads = []
for i in range(len(url_list)):
    t = threading.Thread(target= file_download, args= (url_list[i], data_list[i])) # passing argument
    t.start()
    threads.append(t)

for thread in threads:
    thread.join()

end = time.perf_counter()
print(f"the program finished in {round(end-start, 2)} second")
    
    

Downloaded: dummy.pdf
Error downloading https://file-examples.com/storage/fe6d74a6/file-example_PNG_500kB.png: HTTP Error 403: Forbidden
Error downloading https://file-examples.com/storage/fe5c9c62f/file-example_TXT_10kB.txt: HTTP Error 403: Forbidden
the program finished in 1.67 second


In [ ]:
# Multithradin using concurrent.features >> make code consise 


# how we can use multithreading for downloads : 
import time 
import concurrent.futures

start = time.perf_counter()

url_list = [
    "https://www.w3.org/WAI/ER/tests/xhtml/testfiles/resources/pdf/dummy.pdf",     # Small PDF file
    "https://file-examples.com/storage/fe5c9c62f/file-example_TXT_10kB.txt",       # 10KB TXT file
    "https://file-examples.com/storage/fe6d74a6/file-example_PNG_500kB.png"        # Small PNG image
]

data_list = ["dummy.pdf", "sample.txt", "image.png"]


import urllib.request
def file_download(url, filename):
    try:
        urllib.request.urlretrieve(url, filename)
        print(f"Downloaded: {filename}")
    except Exception as e:
        print(f"Error downloading {url}: {e}")

# Use ThreadPoolExecutor
with concurrent.futures.ThreadPoolExecutor() as executor:
    executor.map(file_download, url_list, data_list)


end = time.perf_counter()
print(f"the program finished in {round(end-start, 2)} second")
    
    

Error downloading https://file-examples.com/storage/fe5c9c62f/file-example_TXT_10kB.txt: HTTP Error 403: Forbidden
Downloaded: dummy.pdf
Error downloading https://file-examples.com/storage/fe6d74a6/file-example_PNG_500kB.png: HTTP Error 403: Forbidden
the program finished in 3.36 second


In [13]:
# share variable across all the threads 
start = time.perf_counter()
shared_counter = 0 
counter_lock = threading.Lock() # locking the counter for a specific therad

def increment_shared_counter(x):
    global shared_counter
    with counter_lock:
        shared_counter = shared_counter+1
        print(f"Thread {x} incremented shared counter to {shared_counter}")
        time.sleep(1)
        
threads = [threading.Thread(target= increment_shared_counter, args= (i,)) for i in [1,2,3,4,5,6]]

for thread in threads:
    thread.start()

for thread in threads:
    thread.join()
    
end = time.perf_counter()

print(f"the program finished in {round(end-start, 2)} seconds.")

Thread 1 incremented shared counter to 1
Thread 2 incremented shared counter to 2
Thread 3 incremented shared counter to 3
Thread 4 incremented shared counter to 4
Thread 5 incremented shared counter to 5
Thread 6 incremented shared counter to 6
the program finished in 6.03 seconds.


In [ ]:
# share variable across all the threads : using concurrent.feature

start = time.perf_counter()
shared_counter = 0 
counter_lock = threading.Lock() # locking the counter for a specific therad

def increment_shared_counter(x):
    global shared_counter
    with counter_lock:
        shared_counter = shared_counter+1
        print(f"Thread {x} incremented shared counter to {shared_counter}")
        time.sleep(1)
        

with concurrent.futures.ThreadPoolExecutor() as executor:
    thread_args = [1,2,3,4,5,6]
    executor.map(increment_shared_counter, thread_args)

end = time.perf_counter()

print(f"the program finished in {round(end-start, 2)} seconds.")

Thread 1 incremented shared counter to 1
Thread 2 incremented shared counter to 2
Thread 3 incremented shared counter to 3
Thread 4 incremented shared counter to 4
Thread 5 incremented shared counter to 5
Thread 6 incremented shared counter to 6
the program finished in 6.03 seconds.
